In [1]:
import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns


def run_phase_5():
  print("==================================================")
  print("      RUNNING PHASE 5: EWMA VOLATILITY SCALING    ")
  print("==================================================")

  # ---------------------------------------------------------
  # 1. Load Phase 3 Data
  # ---------------------------------------------------------
  input_file = "portfolio_phase3_output(10).csv"
  if not os.path.exists(input_file):
    raise FileNotFoundError(
        f"'{input_file}' not found! Please run Phase 3 first."
    )

  df_p3 = pd.read_csv(input_file, parse_dates=["Date"], index_col="Date")
  returns = df_p3["Daily_Return"]
  pnl = df_p3["Daily_PnL"]
  dates = df_p3.index

  # Parameters
  lam = 0.94  # Decay factor for daily returns
  window = 250  # 250-day lookback window
  conf_level = 0.99  # 99% confidence level
  alpha = 1.0 - conf_level

  # ---------------------------------------------------------
  # 2. Calculate EWMA Variance & Volatility
  # ---------------------------------------------------------
  # Shift returns by 1 day to prevent lookahead bias
  squared_returns_shifted = (returns.shift(1) ** 2).fillna(
      returns.iloc[:30].var()
  )

  # Recursively apply EWMA formula via pandas ewm
  ewma_var = squared_returns_shifted.ewm(alpha=1 - lam, adjust=False).mean()
  ewma_vol = np.sqrt(ewma_var)

  # ---------------------------------------------------------
  # 3. Scale Historical Returns & Compute Rolling VaR
  # ---------------------------------------------------------
  raw_var_series = pd.Series(index=dates, dtype=float)
  scaled_var_series = pd.Series(index=dates, dtype=float)

  # Latest day rescaled return series for visualization (using final date T)
  final_vol = ewma_vol.iloc[-1]
  scaled_returns_latest = returns * (final_vol / ewma_vol)

  print(" Calculating rolling EWMA-scaled VaR series...")
  for i in range(window, len(dates)):
    curr_vol = ewma_vol.iloc[i]

    # Slice 250-day historical window
    win_pnl = pnl.iloc[i - window : i]
    win_vols = ewma_vol.iloc[i - window : i]

    # A. Raw Historical VaR (Dollar)
    raw_var = -np.percentile(win_pnl, alpha * 100)
    raw_var_series.iloc[i] = raw_var

    # B. EWMA-Scaled Historical VaR (Dollar)
    scaled_pnl = win_pnl * (curr_vol / win_vols)
    scaled_var = -np.percentile(scaled_pnl, alpha * 100)
    scaled_var_series.iloc[i] = scaled_var

  # ---------------------------------------------------------
  # 4. Exception / Breach Analysis
  # ---------------------------------------------------------
  actual_loss = -pnl
  raw_exceptions = (actual_loss > raw_var_series) & raw_var_series.notna()
  scaled_exceptions = (
      actual_loss > scaled_var_series
  ) & scaled_var_series.notna()

  valid_count = raw_var_series.dropna().count()
  print("\n--- VaR Exception (Breach) Comparison ---")
  print(
      f"Total Evaluated Days:            {valid_count} days"
  )
  print(
      f"Raw 99% VaR Breaches:            {raw_exceptions.sum()} "
      f"({raw_exceptions.sum()/valid_count:.2%})"
  )
  print(
      f"EWMA-Scaled 99% VaR Breaches:    {scaled_exceptions.sum()} "
      f"({scaled_exceptions.sum()/valid_count:.2%})"
  )
  print(f"Expected Breaches (1% Target):   {int(valid_count * 0.01)} days")

  # Save Summary to CSV
  phase5_output = pd.DataFrame(
      {
          "EWMA_Volatility": ewma_vol,
          "Raw_99_VaR": raw_var_series,
          "Scaled_99_VaR": scaled_var_series,
          "Raw_Breach": raw_exceptions,
          "Scaled_Breach": scaled_exceptions,
      },
      index=dates,
  )
  phase5_output.to_csv("ewma_scaled_var_summary_phase5(10).csv")
  print(" [✓] Saved Phase 5 Data -> ewma_scaled_var_summary_phase5(10).csv")

  # ---------------------------------------------------------
  # 5. Generate & Save Required Charts
  # ---------------------------------------------------------
  sns.set_theme(style="whitegrid")

  # --- CHART 1: EWMA Volatility Time Series ---
  plt.figure(figsize=(12, 5))
  plt.plot(
      ewma_vol.index,
      ewma_vol * np.sqrt(252),
      color="darkblue",
      lw=1.5,
      label="Annualized EWMA Volatility (λ = 0.94)",
  )
  plt.title("Chart 1: Portfolio EWMA Annualized Volatility Time Series")
  plt.xlabel("Date")
  plt.ylabel("Annualized Volatility (%)")
  plt.legend()
  plt.tight_layout()
  plt.savefig("chart1_ewma_volatility(10).png", dpi=300)
  plt.close()
  print(" [✓] Saved -> chart1_ewma_volatility(10).png")

  # --- CHART 2: Raw Returns vs Volatility-Scaled Returns ---
  plt.figure(figsize=(12, 5))
  plt.plot(
      returns.index,
      returns,
      color="lightgray",
      alpha=0.7,
      lw=0.8,
      label="Raw Returns",
  )
  plt.plot(
      returns.index,
      scaled_returns_latest,
      color="crimson",
      alpha=0.6,
      lw=0.8,
      label="Volatility-Scaled Returns (to Today's Vol)",
  )
  plt.title("Chart 2: Raw Daily Returns vs. Volatility-Scaled Daily Returns")
  plt.xlabel("Date")
  plt.ylabel("Daily Return")
  plt.legend()
  plt.tight_layout()
  plt.savefig("chart2_raw_vs_scaled_returns(10).png", dpi=300)
  plt.close()
  print(" [✓] Saved -> chart2_raw_vs_scaled_returns(10).png")

  # --- CHART 3: Historical VaR vs EWMA-Scaled VaR ---
  plt.figure(figsize=(12, 6))
  plt.plot(
      raw_var_series.index,
      raw_var_series,
      label="Raw 99% Historical VaR ($)",
      color="darkorange",
      lw=1.5,
      linestyle="--",
  )
  plt.plot(
      scaled_var_series.index,
      scaled_var_series,
      label="EWMA-Scaled 99% Historical VaR ($)",
      color="purple",
      lw=1.8,
  )
  plt.title(
      "Chart 3: 250-Day Rolling Historical VaR vs. EWMA Volatility-Scaled VaR"
  )
  plt.xlabel("Date")
  plt.ylabel("1-Day Potential Dollar Loss ($)")
  plt.legend()
  plt.tight_layout()
  plt.savefig("chart3_historical_vs_ewma_var(10).png", dpi=300)
  plt.close()
  print(" [✓] Saved -> chart3_historical_vs_ewma_var(10).png")

  # --- CHART 4: Exception Comparison Chart ---
  plt.figure(figsize=(12, 6))
  plt.plot(
      actual_loss.index,
      actual_loss,
      color="silver",
      alpha=0.5,
      lw=0.8,
      label="Actual Daily Portfolio Loss ($)",
  )
  plt.plot(
      raw_var_series.index,
      raw_var_series,
      color="darkorange",
      lw=1.2,
      linestyle="--",
      label="Raw 99% VaR Threshold",
  )
  plt.plot(
      scaled_var_series.index,
      scaled_var_series,
      color="purple",
      lw=1.5,
      label="EWMA-Scaled 99% VaR Threshold",
  )

  # Highlight Breaches
  raw_breach_pts = actual_loss[raw_exceptions]
  scaled_breach_pts = actual_loss[scaled_exceptions]

  plt.scatter(
      raw_breach_pts.index,
      raw_breach_pts,
      color="red",
      s=30,
      zorder=5,
      label=f"Raw VaR Breaches ({raw_exceptions.sum()})",
  )
  plt.scatter(
      scaled_breach_pts.index,
      scaled_breach_pts,
      color="blue",
      marker="x",
      s=40,
      zorder=6,
      label=f"Scaled VaR Breaches ({scaled_exceptions.sum()})",
  )

  plt.title(
      "Chart 4: VaR Exceptions / Breaches Comparison (Actual Loss vs. VaR"
      " Thresholds)"
  )
  plt.xlabel("Date")
  plt.ylabel("Dollar Loss ($)")
  plt.legend()
  plt.tight_layout()
  plt.savefig("chart4_exception_comparison(10).png", dpi=300)
  plt.close()
  print(" [✓] Saved -> chart4_exception_comparison(10).png")

  print("\n==================================================")
  print("          PHASE 5 COMPLETED SUCCESSFULLY!         ")
  print("==================================================")


if __name__ == "__main__":
  run_phase_5()

      RUNNING PHASE 5: EWMA VOLATILITY SCALING    
 Calculating rolling EWMA-scaled VaR series...

--- VaR Exception (Breach) Comparison ---
Total Evaluated Days:            2262 days
Raw 99% VaR Breaches:            47 (2.08%)
EWMA-Scaled 99% VaR Breaches:    36 (1.59%)
Expected Breaches (1% Target):   22 days
 [✓] Saved Phase 5 Data -> ewma_scaled_var_summary_phase5.csv
 [✓] Saved -> chart1_ewma_volatility(10).png
 [✓] Saved -> chart2_raw_vs_scaled_returns(10).png
 [✓] Saved -> chart3_historical_vs_ewma_var(10).png
 [✓] Saved -> chart4_exception_comparison(10).png

          PHASE 5 COMPLETED SUCCESSFULLY!         
